In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch
from omegaconf import OmegaConf

from src.builder import build_critic, build_generator, validate_config

In [2]:
cfg = OmegaConf.load(REPO_ROOT / "configs" / "default.yaml")
validate_config(cfg)

B = 2
C = cfg.data.in_channels
D, H, W = cfg.data.train_shape

netG = build_generator(cfg).eval()
netC = build_critic(cfg).eval()

In [3]:
sparse = torch.zeros(B, C, D, H, W)
mask = torch.zeros(B, 1, D, H, W)

with torch.no_grad():
    fake = netG(sparse, mask)

print("sparse :", tuple(sparse.shape))
print("mask   :", tuple(mask.shape))
print("fake   :", tuple(fake.shape))

sparse : (2, 1, 64, 64, 64)
mask   : (2, 1, 64, 64, 64)
fake   : (2, 1, 64, 64, 64)


In [4]:
slices_axis0 = fake[:, :, 0, :, :]
slices_axis1 = fake[:, :, :, 0, :]
slices_axis2 = fake[:, :, :, :, 0]

with torch.no_grad():
    score0 = netC(slices_axis0)
    score1 = netC(slices_axis1)
    score2 = netC(slices_axis2)

print("axis 0 slice :", tuple(slices_axis0.shape), "-> score", tuple(score0.shape))
print("axis 1 slice :", tuple(slices_axis1.shape), "-> score", tuple(score1.shape))
print("axis 2 slice :", tuple(slices_axis2.shape), "-> score", tuple(score2.shape))

axis 0 slice : (2, 1, 64, 64) -> score (2, 1, 1, 1)
axis 1 slice : (2, 1, 64, 64) -> score (2, 1, 1, 1)
axis 2 slice : (2, 1, 64, 64) -> score (2, 1, 1, 1)
